In [ ]:
import plotly.express as px
import pandas as pd
import struct
from glob import glob
from os.path import basename

def summarize_all_componentwise_errors(matrix_type):

    filepaths = glob(f"results/{matrix_type}/all_componentwise_errors.*.bin")
    
    dfs = []
    for filepath in filepaths:
        with open(filepath, "rb") as f:
            byte_buffer = f.read()

        errors = []
        for i in range(0,len(byte_buffer),8):
            errors.append(struct.unpack('d', byte_buffer[i:i+8]))

        df = pd.DataFrame(errors, columns=['error'])
        df["filename"] = basename(filepath)
        dfs.append(df)

    df = pd.concat(dfs)
    for filepath in filepaths:
        df_slice = df[df['filename'] == basename(filepath)]
        print(f"{filepath} stats:")
        print(f"\t   mean: {df_slice['error'].mean()}")
        print(f"\t    std: {df_slice['error'].std()}")

    fig = px.box(
        df,
        x='filename',
        y='error',
        points=False,
    )
    fig.update_layout(
        height = 1000
    )
    fig.update_layout(
        yaxis_tickformat='.2e',
    )
    fig.show()

'''
    fig = px.histogram(
        df,
        x='error',
        color='filename',
        log_y=True,
        title=f"All Componentwise Errors for {matrix_type} matrices"
    )
    fig.update_traces(opacity=0.5)
    fig.update_layout(yaxis_tickformat='.2e', xaxis_tickformat='.2e')
    fig.show()
'''


def plot_max_componentwise_errors_vs_flops(matrix_type):

    filepaths = glob(f"results/{matrix_type}/max_componentwise_abs_errors.*.txt")
    
    dfs = []
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        df["filename"] = basename(filepath)
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)

    df["FLOPs"] = 2 * df["m"] * df["n"] * df["k"]
    df["Square?"] = (df["m"] == df["n"]) & (df["n"] == df["k"])
    fig = px.scatter(
        df,
        x="FLOPs",
        y="max_componentwise_abs_error",
        log_x=True,
        log_y=True,
        title=f"Max Componentwise Error vs FLOPs for {matrix_type} matrices",
        hover_data=["m", "n", "k"],
        symbol="Square?",
        symbol_sequence=["square", "arrow-wide"],
        color="filename",
    )
    fig.update_traces(
        marker=dict(
            size=15,
            opacity=0.7,
        )
    )
    fig.update_layout(
        yaxis_tickformat='.2e',
        xaxis_tickformat='.2e'
    )

    fig.show()

def count_n_zeros(matrix_type):
    filepaths = glob(f"results/{matrix_type}/max_componentwise_abs_errors.*.txt")
    
    dfs = []
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        df["filename"] = basename(filepath)
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)

    for filepath in filepaths:
        df_slice = df[df['filename'] == basename(filepath)]
        print(f"{filepath}:")
        print(f"\tn_zeros: {df_slice['n_zeros'].sum()}\n")



In [ ]:
matrix_types = ["test1"]

In [ ]:
for matrix_type in matrix_types:
    summarize_all_componentwise_errors(matrix_type)

In [ ]:
for matrix_type in matrix_types:
    plot_max_componentwise_errors_vs_flops(matrix_type)

In [ ]:
for matrix_type in matrix_types:
    count_n_zeros(matrix_type)